In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.sql import functions as F

# Sample Data

rows = [
    Row(
        id=1,
        names = ['Alice','Bob'],
        scores = [85,60],
        info = Row(age = 25, city = 'New York'),
        json_str = '{"dept":"Engineering", "level":2}'
    ),
    Row(
        id=2,
        names = ['Charlie','Dave',"Mark"],
        scores = [90,70,58],
        info = Row(age = 30, city = 'San Francisco'),
        json_str = '{"dept":"Marketing", "level":3}'
    )
]


schema = StructType([
    StructField("Id", IntegerType(),True),
    StructField("names", ArrayType(StringType()), True),
    StructField("scores", ArrayType(IntegerType()), True),
    StructField("info", StructType([
        StructField("age", IntegerType(), True),
        StructField("city", StringType(), True)
    ]), True),
    StructField("json_str", StringType(), True)
])

df = spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("df")
display(df)


In [0]:
from pyspark.sql import functions as F

# Get the length of the 'names' array
array_len_df = df.withColumn("name_length",F.size(F.col("names")))
 # check if "names" array contains 'Alice'
array_contains_df = array_len_df.withColumn("has_alice",F.array_contains(F.col("names"),"Alice"))

# split a string column into array ( for demonstration, create a new columnn)

df_array = array_contains_df.withColumn("city_words",F.split(F.col("info.city")," "))


display(df_array.select('Id','names','name_length','has_alice','city_words'))

In [0]:
exploded_df = df.select("Id",F.explode(F.col("names")).alias("name"))

display(exploded_df)

In [0]:
 # Struct Type where multiple array stored in a single column
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# accesing fields in the info struct
struct_df = df.withColumn('age',F.col("info.age")).withColumn("city",F.col("info.city"))

display(struct_df)




In [0]:
# Struct are object store in array and Json type are string s
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Extract a value using get_json_object
json_df = df.withColumn("dept",F.get_json_object(F.col("json_str"),"$.dept")).withColumn("level",F.get_json_object(F.col("json_str"),"$.level"))

display(json_df)


In [0]:
# Parse json string into struct


json_schema = StructType([
    StructField("dept",StringType(), True),
    StructField("level",IntegerType(), True)

    ]) 

from_json_df = df.withColumn("json_struct",F.from_json(F.col("json_str"), json_schema))

# Accessing fields in the 'info' struct

struct_df = from_json_df.withColumn("dept", F.col("json_struct.dept"))

display(struct_df)


